In [1]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import notebooks_DL_functions as DL

In [2]:
nn_architecture = [
    {"input_dim": 2, "output_dim": 4, "activation": "relu"},
    {"input_dim": 4, "output_dim": 6, "activation": "relu"},
    {"input_dim": 6, "output_dim": 6, "activation": "relu"},
    {"input_dim": 6, "output_dim": 4, "activation": "relu"},
    {"input_dim": 4, "output_dim": 1, "activation": "sigmoid"}

]

In [35]:
def init_layer(nn_architecture, seed=99):
  np.random.seed(seed)
  params_value = {}
  for idx, layer in enumerate(nn_architecture):
    layer_idx = idx + 1
    params_value["W" + str(layer_idx)] = np.random.randn(
        layer["output_dim"], layer["input_dim"]) * 0.1
    params_value["b" + str(layer_idx)] = np.zeros((layer["output_dim"], 1))

  return params_value

In [4]:
init_layer(nn_architecture)

{'W1': array([[-0.01423588,  0.20572217],
        [ 0.02832619,  0.1329812 ],
        [-0.01546219, -0.00690309],
        [ 0.07551805,  0.08256466]]),
 'b1': array([[0.],
        [0.],
        [0.],
        [0.]])}

In [5]:
def sigmoid_backward(dA, x):
  sig = DL.sigmoid(x)
  return dA * sig * (1 - sig)

In [6]:
def relu_backward(dA, x):
  dx =np.array(dA, copy=True)
  dx[x <= 0] = 0
  return dx

In [7]:
def convert_prob_into_class(probs):
  """COnvert probabilities into binary class (0 or 1)"""
  return (probs > 0.5).astype(int)

In [8]:
import numpy as np

def convert_prob_into_class(probs):
  """Convert probabilities into binary class (0 or 1)"""
  return (np.array(probs) > 0.5).astype(int)


In [9]:
convert_prob_into_class(0.5)


np.int64(0)

In [10]:
def single_layer_forward_propagation(A_prev, W_curr, b_curr, activation="relu"):
  Z_curr = np.dot(W_curr, A_prev) + b_curr
  if activation == "relu":
    activation_func = DL.relu
  elif activation == "sigmoid":
    activation_func = DL.sigmoid
  else:
    raise Exception("Non supported activation function")
  return activation_func(Z_curr), Z_curr

In [11]:
single_layer_forward_propagation(1, 0.5, -7, "relu")

(0, np.float64(-6.5))

In [21]:
def full_forward_propagation(X, params_values, nn_architecture):
  memory = {}
  A_curr = X
  for idx, layer in enumerate(nn_architecture):
    layer_idx = idx + 1
    A_prev = A_curr
    A_curr, Z_curr = single_layer_forward_propagation(
      A_prev, params_values["W" + str(layer_idx)],
      params_values["b" + str(layer_idx)],
      layer["activation"]
    )
    memory["A" + str(idx)] = A_prev
    memory["Z" + str(layer_idx)] = Z_curr
  return A_curr, memory

In [22]:
def get_cost_value(Y_hat, Y):
  m = Y_hat.shape[1]
  cost = -1 / m * (np.dot(Y, np.log(Y_hat).T) + np.dot(1 - Y, np.log(1 - Y_hat).T))
  return np.squeeze(cost)


In [36]:
def get_accuracy_value(Y_hat, Y):
  Y_hat_class = convert_prob_into_class(Y_hat)
  # Ensure Y has the same shape as Y_hat_class for element-wise comparison
  return (Y_hat_class == Y.T).mean()

In [24]:
def single_layer_backward_propagation(dA_curr, W_curr, b_curr, Z_curr, A_prev, activation="relu"):
  m = A_prev.shape[1]
  if activation == "relu":
    backward_activation_func = relu_backward
  elif activation == "sigmoid":
    backward_activation_func = sigmoid_backward
  else:
    raise Exception("Non supported activation function")

  dZ_curr = backward_activation_func(dA_curr, Z_curr)
  dW_curr = np.dot(dZ_curr, A_prev.T) / m
  db_curr = np.sum(dZ_curr, axis=1, keepdims=True) / m
  dA_prev = np.dot(W_curr.T, dZ_curr)
  return dA_prev, dW_curr, db_curr

In [25]:
def full_backward_propagation(Y_hat, Y, memory, params_values, nn_architecture):
  grads_values = {}
  Y = Y.reshape(Y_hat.shape)
  dA_prev = - (np.divide(Y, Y_hat) - np.divide(1 -Y, 1 - Y_hat))
  for layer_idx_prev, layer in reversed(list(enumerate(nn_architecture))):
    layer_idx_curr = layer_idx_prev + 1
    dA_prev, dW_curr, db_curr = single_layer_backward_propagation(
      dA_prev,
      params_values["W" + str(layer_idx_curr)],
      params_values["b" + str(layer_idx_curr)],
      memory["Z" + str(layer_idx_curr)],
      memory["A" + str(layer_idx_prev)],
      layer["activation"]
    )
    grads_values["dW" + str(layer_idx_curr)] = dW_curr
    grads_values["db" + str(layer_idx_curr)] = db_curr
  return grads_values

In [26]:
from re import A
def train(X, Y, nn_architecture, epochs, learning_rate):
  params_values = init_layer(nn_architecture, 2)
  cost_history = []
  accuracy_history = []
  for i in range(epochs):
    Y_hat, memory = full_forward_propagation(X, params_values, nn_architecture)
    cost = get_cost_value(Y_hat, Y)
    cost_history.append(cost)
    accuracy = get_accuracy_value(Y_hat, Y)
    accuracy_history.append(accuracy)
    grads_values = full_backward_propagation(Y_hat, Y, memory, params_values, nn_architecture)
    params_values = update(params_values, grads_values, nn_architecture, learning_rate)

    if i % 1000 == 0:
      print(f"Iteration: {i} - cost: {cost:.5f} - accuracy: {accuracy:.5f}")
  return params_values, cost_history, accuracy_history

In [38]:
def update(params_values, grads_values, nn_architecture, learning_rate):
  for idx, layer in enumerate(nn_architecture):
    layer_idx = idx + 1
    params_values["W" + str(layer_idx)] -= learning_rate * grads_values["dW" + str(layer_idx)]
    params_values["b" + str(layer_idx)] -= learning_rate * grads_values["db" + str(layer_idx)]
  return params_values

In [39]:
def train(X, Y, nn_architecture, epochs, learning_rate):
  params_values = init_layer(nn_architecture, 2)
  cost_history = []
  accuracy_history = []

  for i in range(epochs):
    Y_hat, memory = full_forward_propagation(X, params_values, nn_architecture)
    cost = get_accuracy_value(Y_hat, Y)
    cost_history.append(cost)
    accuracy = get_accuracy_value(Y_hat, Y)
    accuracy_history.append(accuracy)
    grads_values = full_backward_propagation(Y_hat, Y, memory, params_values, nn_architecture)
    params_values = update(params_values, grads_values, nn_architecture, learning_rate)

    if i % 1000 == 0:
      print(f"Iteration: {i} - cost: {cost:.5f} - accuracy: {accuracy:.5f}")

  return params_values, cost_history, accuracy_history


In [40]:
X, y = make_moons(n_samples=1000, noise=0.1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_T = X_train.T
y_train_T = y_train.reshape(-1, 1)


params, cost_h, acc_h = train(X_train_T, y_train_T, nn_architecture, 10000, 0.01)

plt.plot(cost_h)
plt.title("Cost history")
plt.show()
plt.plot(acc_h)
plt.title("Accuracy history")
plt.show()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [34]:
X_train_T = X_train.T
y_train_T = y_train.reshape(-1, 1)